In [3]:
import os
import psycopg2
import pandas as pd
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Database connection parameters
db_params = {
    'dbname': 'dbt_equipment_losses',
    'user': os.getenv('DBT_USER'),
    'password': os.getenv('DBT_PASS'),
    'host': 'localhost',
    'port': 5432
}

# Establish a connection to the database
conn = psycopg2.connect(**db_params)

# Create a cursor object
cur = conn.cursor()

# Execute a SELECT query
cur.execute("SELECT * FROM public.mart_equipment_analysis ORDER BY predicted_category, date_recorded")

# Fetch all rows from the result
rows = cur.fetchall()

# Get column names
column_names = [desc[0] for desc in cur.description]

# Create a pandas DataFrame
df = pd.DataFrame(rows, columns=column_names)

# Close the cursor and connection
cur.close()
conn.close()

# Display the first few rows of the DataFrame
print(df.head())

# You can now work with the DataFrame 'df' which contains your data
# For example, you can perform analysis, create visualizations, etc.

# To save the DataFrame to a CSV file:
# df.to_csv('equipment_analysis.csv', index=False)

  predicted_category  total_losses  destroyed  captured  damaged  \
0           Aircraft           253        230         1       22   
1           Aircraft           253        230         1       22   
2           Aircraft           253        230         1       22   
3           Aircraft           253        230         1       22   
4           Aircraft           253        230         1       22   

  date_recorded cumulative_losses  
0    2022-03-19                24  
1    2022-03-26                28  
2    2022-03-28                29  
3    2022-04-04                32  
4    2022-04-06                39  


In [5]:
import hvplot.pandas

# 1. Time series plot of equipment losses by category
time_series_plot = df.hvplot.line(
    x='date_recorded', 
    y='equipment_losses', 
    by='predicted_category', 
    title='Equipment Losses Over Time by Category',
    height=400, 
    width=800
)

# 2. Bar chart of total losses by category
total_losses_by_category = df.groupby('predicted_category')['equipment_losses'].sum().reset_index()
bar_chart = total_losses_by_category.hvplot.bar(
    x='predicted_category', 
    y='equipment_losses', 
    title='Total Equipment Losses by Category',
    height=400, 
    width=600
)

# 3. Scatter plot of losses vs. confidence score
scatter_plot = df.hvplot.scatter(
    x='confidence_score', 
    y='equipment_losses', 
    by='predicted_category',
    title='Equipment Losses vs. Confidence Score',
    height=400, 
    width=600
)

# 4. Heatmap of losses over time
heatmap = df.hvplot.heatmap(
    x='date_recorded', 
    y='predicted_category', 
    C='equipment_losses', 
    colorbar=True,
    title='Heatmap of Equipment Losses Over Time by Category',
    height=400, 
    width=800
)

# Display the plots
time_series_plot
bar_chart
scatter_plot
heatmap


DataError: Supplied data does not contain specified dimensions, the following dimensions were not found: ['equipment_losses']

PandasInterface expects tabular data, for more information on supported datatypes see https://holoviews.org/user_guide/Tabular_Datasets.html